# Self-Attention from Scratch — with Real Numbers

Three tiny steps, each computed by hand-sized numbers you can check on paper:

1. **Attention with no weights:** `softmax(X Xᵀ) X`
2. **Learnable queries, keys and values:** `softmax(Q Kᵀ / √d_k) V`
3. **Multi-head attention:** several heads, concatenated

Running example: the sentence **"bank of river"**, with 4-dimensional toy embeddings.

This notebook comes from Sessions 2–4 of the course
[Master Transformers from Scratch: BERT, ViT & AI Search](https://www.udemy.com/course/master-transformers-from-scratch-bert-vit-ai-search/).

In [1]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

def softmax(scores):
    shifted = scores - scores.max(axis=-1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=-1, keepdims=True)

X = np.array([
    [1, 0, 1, 0],   # bank
    [0, 1, 0, 1],   # of
    [1, 1, 0, 0],   # river
], dtype=float)
tokens = ["bank", "of", "river"]
X.shape  # (n tokens, d dimensions)

(3, 4)

## Step 1 — Score, normalize, mix

* **Score:** `S = X Xᵀ`. Entry `(i, j)` is the dot product of token *i* with token *j*: how similar they are.
* **Normalize:** `A = softmax(S)`, row by row. Each row is a *focus recipe* that sums to 1.
* **Mix:** `Y = A X`. Each new token is a weighted average of all the old ones.

In [2]:
S = X @ X.T
print("Scores S:\n", S)

Scores S:
 [[2. 0. 1.]
 [0. 2. 1.]
 [1. 1. 2.]]


In [3]:
A = softmax(S)
print("Focus matrix A:\n", A)
print("Row sums:", A.sum(axis=1))

Focus matrix A:
 [[0.665 0.09  0.245]
 [0.09  0.665 0.245]
 [0.212 0.212 0.576]]
Row sums: [1. 1. 1.]


In [4]:
Y = A @ X
for t, before, after in zip(tokens, X, Y):
    print(f"{t:>6}: {before} -> {after}")

  bank: [1. 0. 1. 0.] -> [0.91  0.335 0.665 0.09 ]
    of: [0. 1. 0. 1.] -> [0.335 0.91  0.09  0.665]
 river: [1. 1. 0. 0.] -> [0.788 0.788 0.212 0.212]


**What happened:** "bank" kept most of itself (0.665) but pulled in 0.245 of "river" and only 0.090 of "of".
Its new vector now carries a little river-ness, which is exactly the context we wanted.

**The problem:** there is nothing to learn here. The focus pattern is fixed by the raw embeddings.

## Step 2 — Make attention learnable with Q, K, V

Give each token three roles, each with its own weight matrix:

* **Query** `Q = X W_Q`: what am I looking for?
* **Key** `K = X W_K`: what do I contain?
* **Value** `V = X W_V`: what do I hand over if selected?

Then `Attention(Q, K, V) = softmax(Q Kᵀ / √d_k) V`. Dividing by `√d_k` stops large dot products from saturating the softmax.

In [5]:
W_Q = np.array([[1, 0, 1], [0, 1, 0], [0, 0, 1], [1, 1, 0]], dtype=float)
W_K = np.array([[0, 1, 1], [1, 0, 0], [1, 0, 1], [0, 1, 0]], dtype=float)
W_V = np.array([[1, 0, 0], [0, 0, 1], [0, 1, 0], [1, 0, 1]], dtype=float)

Q, K, V = X @ W_Q, X @ W_K, X @ W_V
print("Q:\n", Q, "\nK:\n", K, "\nV:\n", V)

Q:
 [[1. 0. 2.]
 [1. 2. 0.]
 [1. 1. 1.]] 
K:
 [[1. 1. 2.]
 [1. 1. 0.]
 [1. 1. 1.]] 
V:
 [[1. 1. 0.]
 [1. 0. 2.]
 [1. 0. 1.]]


In [6]:
d_k = K.shape[1]
S = Q @ K.T
S_scaled = S / np.sqrt(d_k)
A = softmax(S_scaled)
Y = A @ V
print("Scores S:\n", S)
print("Scaled by sqrt(3):\n", S_scaled)
print("Focus matrix A:\n", A)
print("Output Y:\n", Y)

Scores S:
 [[5. 1. 3.]
 [3. 3. 3.]
 [4. 2. 3.]]
Scaled by sqrt(3):
 [[2.887 0.577 1.732]
 [1.732 1.732 1.732]
 [2.309 1.155 1.732]]
Focus matrix A:
 [[0.707 0.07  0.223]
 [0.333 0.333 0.333]
 [0.533 0.168 0.299]]
Output Y:
 [[1.    0.707 0.363]
 [1.    0.333 1.   ]
 [1.    0.533 0.635]]


Compare the "bank" row: Step 1 gave `[0.665, 0.090, 0.245]`, Step 2 gives `[0.707, 0.070, 0.223]`.
The weight matrices changed the focus pattern. During training, gradients adjust `W_Q`, `W_K` and `W_V`
until the focus pattern helps the task.

## Step 3 — Multi-head attention

One head learns one kind of relationship. With several heads, each gets its own small `W_Q`, `W_K`, `W_V`
and can look at the sentence differently. Their outputs are concatenated and projected by `W_O`.

Here: 2 heads, `d_model = 4`, `d_k = 2`, and `W_O` is the identity so you can read the raw head outputs.

In [7]:
def attention(X, W_Q, W_K, W_V):
    Q, K, V = X @ W_Q, X @ W_K, X @ W_V
    A = softmax(Q @ K.T / np.sqrt(K.shape[1]))
    return A @ V, A

head1 = ([[1, 0], [0, 1], [1, 0], [0, 1]], [[0, 1], [1, 0], [0, 1], [1, 0]], [[1, 0], [0, 1], [0, 1], [1, 0]])
head2 = ([[0, 1], [1, 0], [1, 1], [0, 0]], [[1, 1], [0, 0], [1, 0], [0, 1]], [[0, 1], [1, 0], [1, 1], [0, 0]])

out1, A1 = attention(X, *map(np.array, head1))
out2, A2 = attention(X, *map(np.array, head2))
print("Head 1 focus for 'bank':", A1[0])
print("Head 2 focus for 'bank':", A2[0])

W_O = np.eye(4)
output = np.concatenate([out1, out2], axis=1) @ W_O
print("Final output:\n", output)

Head 1 focus for 'bank': [0.045 0.768 0.187]
Head 2 focus for 'bank': [0.576 0.14  0.284]
Final output:
 [[1.    1.    1.    1.436]
 [1.    1.    1.    1.436]
 [1.    1.    1.    1.436]]


Head 1 makes "bank" look mostly at "of" (0.768). Head 2 makes "bank" look mostly at itself (0.576), then "river".
Different heads, different perspectives, and the output has the **same shape as the input**, so these blocks can be stacked.

## The same thing in PyTorch

`self_attention_torch.py` in this repo wraps all of this in `nn.Module` classes that work on batches and can be trained.

In [8]:
import torch, math

class MultiHeadAttention(torch.nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.h, self.d_k = num_heads, d_model // num_heads
        self.W_Q = torch.nn.Linear(d_model, d_model, bias=False)
        self.W_K = torch.nn.Linear(d_model, d_model, bias=False)
        self.W_V = torch.nn.Linear(d_model, d_model, bias=False)
        self.W_O = torch.nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        b, n, d = x.shape
        split = lambda t: t.view(b, n, self.h, self.d_k).transpose(1, 2)
        Q, K, V = split(self.W_Q(x)), split(self.W_K(x)), split(self.W_V(x))
        A = (Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)).softmax(-1)
        return self.W_O((A @ V).transpose(1, 2).reshape(b, n, d))

x = torch.randn(2, 5, 64)
print(MultiHeadAttention(64, 8)(x).shape)

torch.Size([2, 5, 64])


## Where to go next

This is the core of every transformer. The full course builds on it step by step: positional encoding,
the encoder block, BERT pretraining in PyTorch, fine-tuning for classification and NER, Vision Transformers,
and two-tower semantic search.

👉 [Master Transformers from Scratch: BERT, ViT & AI Search](https://www.udemy.com/course/master-transformers-from-scratch-bert-vit-ai-search/)